<a href="https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Answer:

**Lane 4 (CTR opportunity scoring), same data as ML-04 and ML-07:** March 2026 warehouse data, 75,817 pages from 37 clients. Clues come from 1–15 March. The label `still_under` = the page was still below the typical CTR for its position band in 16–31 March.

**Task type:** ranking ("which pages should an editor review first?"). So I use models that output a **probability** that a page will stay under-performing, rank pages by that probability, and measure **Precision@K**, the same metric as my Week-4 baseline.

**What I compare:**
1. **Week-4 baseline rule (frozen, unchanged):** top-20 position, at least 1,000 impressions, CTR below typical, ranked by missed clicks. No training.
2. **Logistic regression:** the simplest learned model. It gives each feature one weight, so I can read which way each feature pushes. Before fitting, I squash the two very skewed features (impressions and missed clicks) with a log so a few giant pages do not dominate, and I put all features on the same scale.
3. **Random forest:** 200 decision trees that vote. It can capture combinations of features (for example, "low CTR matters more for big pages") that one weight per feature cannot.

**Why not a single decision tree:** a small tree gives only a few distinct scores, so many pages tie at the top and Precision@K becomes partly luck (I saw this in notebook 02).

**Why not clustering or gradient boosting:** clustering groups pages but does not answer "which first?". Gradient boosting is a stronger relative of the random forest; I would only try it if the forest clearly beat the baseline. I prefer the simplest method that wins.

**Features:** the same five as my data contract, all from 1–15 March: `impressions_h1`, `ctr_h1`, `position_h1`, `days_visible_h1`, `missed_clicks_h1`.

In [1]:
%pip -q install duckdb

import duckdb
import numpy as np
import pandas as pd
import sklearn
from google.colab import userdata

# Connect to the warehouse
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Clues from 1-15 March, answer from 16-31 March.
pages = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS days_visible_h1,
        SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15') AS impressions_h2,
        SUM(gsc_clicks)      FILTER (WHERE report_date > DATE '2026-03-15') AS clicks_h2,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h2
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING impressions_h1 >= 100 AND impressions_h2 >= 50
""").df()
pages = pages.dropna(subset=["position_h1", "position_h2"]).copy()

bands = [0, 3, 5, 7, 10, 15, 20, 30, 50, 1000]

# Clues (first half only)
pages["ctr_h1"] = pages["clicks_h1"] / pages["impressions_h1"] * 100
pages["band_h1"] = pd.cut(pages["position_h1"], bands)
pages["expected_ctr_h1"] = pages.groupby("band_h1", observed=True)["ctr_h1"].transform("median")
pages["missed_clicks_h1"] = pages["impressions_h1"] * (pages["expected_ctr_h1"] - pages["ctr_h1"]) / 100

# Answer (second half only)
pages["ctr_h2"] = pages["clicks_h2"] / pages["impressions_h2"] * 100
pages["band_h2"] = pd.cut(pages["position_h2"], bands)
pages["expected_ctr_h2"] = pages.groupby("band_h2", observed=True)["ctr_h2"].transform("median")
pages["still_under"] = (pages["ctr_h2"] < pages["expected_ctr_h2"]).astype(int)

# Week-4 baseline rule
MIN_IMPRESSIONS, MAX_POSITION = 1000, 20
flagged = (
    (pages["impressions_h1"] >= MIN_IMPRESSIONS) &
    (pages["position_h1"] <= MAX_POSITION) &
    (pages["ctr_h1"] < pages["expected_ctr_h1"])
)
pages["baseline_score"] = np.where(flagged, pages["missed_clicks_h1"], 0.0)

FEATURES = ["impressions_h1", "ctr_h1", "position_h1", "days_visible_h1", "missed_clicks_h1"]

print("Pages:", len(pages), "| clients:", pages["client_hash_id"].nunique())
print("Base rate:", round(pages["still_under"].mean(), 2), "| pages flagged by the baseline:", int(flagged.sum()))
print("scikit-learn version:", sklearn.__version__)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 75817 | clients: 37
Base rate: 0.35 | pages flagged by the baseline: 8710
scikit-learn version: 1.6.1


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Answer:

**Design: grouped 5-fold cross-validation by client (`GroupKFold`, 5 folds).** The 37 clients are split into 5 groups. Each time, the models train on 4 groups and are tested on the 5th, whose clients they have never seen. Every client is tested exactly once. The baseline rule needs no training, but it is scored on exactly the same 5 test sets, so all methods face the same "exams" and the same metric.

**Why grouped by client:** pages from the same website share habits (design, brand, tracking, search features). In ML-07, one client filled about half of my top 100. If pages from one client were in both training and testing, a model could learn "this website is bad" and look better than it really is on new websites. The table below shows `clients_in_both` = 0 in every fold, so this cannot happen.

**Time order:** my features come from 1–15 March and my label from 16–31 March, so every prediction uses only information from before the outcome. A split across different months (train on earlier months, test on later ones) would test stability over time; that is a possible next step.

**What the folds look like:** they are uneven. Fold 1 is a single client with 17,425 pages, fold 2 has 3 clients, and folds 3–5 have 11 clients each. The base rate also varies a lot between folds (0.24 to 0.50), so I always compare methods inside the same fold and print that fold's base rate next to them. Because fold 1 is one client, its result describes that one website.

In [2]:
# SPLIT DESIGN: 5 "exams", grouped by client
# Each fold hides about 1/5 of the clients; models train on the rest and are tested on the hidden ones.
from sklearn.model_selection import GroupKFold

folds = list(GroupKFold(n_splits=5).split(pages, groups=pages["client_hash_id"]))

rows = []
for i, (train_idx, test_idx) in enumerate(folds, 1):
    train_clients = set(pages.iloc[train_idx]["client_hash_id"])
    test_clients = set(pages.iloc[test_idx]["client_hash_id"])
    rows.append({
        "fold": i,
        "train_pages": len(train_idx),
        "test_pages": len(test_idx),
        "test_clients": len(test_clients),
        "clients_in_both": len(train_clients & test_clients),   # must be 0
        "test_base_rate": round(pages.iloc[test_idx]["still_under"].mean(), 2),
    })
display(pd.DataFrame(rows))

,fold,train_pages,test_pages,test_clients,clients_in_both,test_base_rate
0,1,58392,17425,1,0,0.44
1,2,61209,14608,3,0,0.50
2,3,61222,14595,11,0,0.26
3,4,61222,14595,11,0,0.32
4,5,61223,14594,11,0,0.24


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# TRAIN + COMPARE: same 5 folds, same metric (Precision@K), same labels for every method
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def squash(df):
    """For logistic regression only: log-squash the two very skewed features."""
    X = df[FEATURES].copy()
    X["impressions_h1"] = np.log1p(X["impressions_h1"])
    X["missed_clicks_h1"] = np.sign(X["missed_clicks_h1"]) * np.log1p(np.abs(X["missed_clicks_h1"]))
    return X

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

KS = [10, 20, 50, 100]
results = []
pages["lr_score"] = np.nan
pages["rf_score"] = np.nan

for i, (train_idx, test_idx) in enumerate(folds, 1):
    train, test = pages.iloc[train_idx], pages.iloc[test_idx]

    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    lr.fit(squash(train), train["still_under"])

    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1)
    rf.fit(train[FEATURES], train["still_under"])

    scores = {
        "Week-4 baseline rule": test["baseline_score"].values,
        "Logistic regression": lr.predict_proba(squash(test))[:, 1],
        "Random forest": rf.predict_proba(test[FEATURES])[:, 1],
    }
    pages.loc[test.index, "lr_score"] = scores["Logistic regression"]
    pages.loc[test.index, "rf_score"] = scores["Random forest"]

    y = test["still_under"].values
    results.append({"fold": i, "method": "Random pick (base rate)", **{f"P@{k}": y.mean() for k in KS}})
    for name, s in scores.items():
        results.append({"fold": i, "method": name, **{f"P@{k}": precision_at_k(s, y, k) for k in KS}})
    print(f"Fold {i} done")

res = pd.DataFrame(results)

print("\nMODEL vs BASELINE - average over the 5 folds (each fold = clients never seen in training):")
display(res.groupby("method", sort=False)[[f"P@{k}" for k in KS]].mean().round(2))

print("\nPrecision@20 in each fold (to see how much results move between folds):")
display(res.pivot(index="fold", columns="method", values="P@20").round(2))

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done

MODEL vs BASELINE - average over the 5 folds (each fold = clients never seen in training):


,P@10,P@20,P@50,P@100
method,,,,
Random pick (base rate),0.35,0.35,0.35,0.35
Week-4 baseline rule,0.94,0.93,0.92,0.90
Logistic regression,0.94,0.92,0.90,0.88
Random forest,0.98,0.98,0.92,0.88



Precision@20 in each fold (to see how much results move between folds):


method,Logistic regression,Random forest,Random pick (base rate),Week-4 baseline rule
fold,,,,
1,0.90,0.95,0.44,0.85
2,0.90,1.00,0.50,0.85
3,0.95,1.00,0.26,1.00
4,0.90,1.00,0.32,0.95
5,0.95,0.95,0.24,1.00


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.